# [DAS509] 6강 데이터 요약의 원리와 적용

### 시각화 원칙

https://clauswilke.com/dataviz/

## 평균 자세히 살펴보기

- 자료요약의 가장 기본이 되는 통계량
- 다음의 데이터를 보자:
    
    $$
    1, 2, 2, 3, 3, 3, 4, 4, 5
    $$
    
- 평균의 계산

$$
\bar x_n = (1 + 2 + 2 + 3 + 3 + 3 + 4+4+5)/9 = 3
$$

In [ ]:
# Example Data
x = np.array([1, 2,2, 3,3,3, 4,4, 5])
# Histogram & Mean
fig, ax = plt.subplots()
ax.hist(x, bins = 5, linewidth = 0.5, edgecolor = "white", alpha = 0.5)
ax.vlines(x.mean(), 0, 3.5, color = "r", linewidth = 2, linestyle = "dashed")
plt.show()

## 분포의 요약

- 평균과 분산은 데이터의 분포를 요약하는 값
- 전체적인 형태를 보기 위해서는 분위수를 활용할 수 있음.
- 최대값 (Minimum), 최소값 (Maximum), 범위 (Max - Min)
- 사분위수 (Quartile): 전체 데이터를 크기순으로 정확히 4등분하는 경계값
    - Q1: 순서대로 나열했을 때, 하위 25% (=상위 75%)에 위치한 값
    - Q2: 순서대로 나열했을 때, 하위 50% (=상위 50%)에 위치한 값 (i.e, 중위수)
    - Q3: 순서대로 나열했을 때, 하위 75% (= 상위 25%)에 위치한 값
- 백분위수 (Percentile): 전체 데이터를 크기순으로 정확히 4등분하는 경계값
- 분위수(Quantile): $100\% \times \alpha$번째  분위수는 데이터를 크기순으로 나열 했을 때, $100\% \times \alpha$번째 위치하는 수 (= 하위 $100\% \times \alpha$)에 위치한 값
- 다섯수치 요약 (Five Number Summary)
    - 최소값, Q1, Q2, Q3, 최대값
    - 상자그림 (Boxplot)은 다섯수치 요약을 시각화 한 그림

In [ ]:
data = np.array(np.random.standard_normal(300))
df = pd.DataFrame(data.reshape(100, 3))

# summary stats
df.describe()

# boxplot
df.boxplot()

- Boston Housing Data

In [ ]:
# boston housing data
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
df = pd.DataFrame(data)
cor = df.corr()
cor.round(2)

In [ ]:
sns.heatmap(cor, cmap="Reds")
plt.show()

- 다양한 형태의 일반화된 상관관계 척도가 있음
    - Spearman Rank Correlation

In [ ]:
cor2 = df.corr(method="spearman")

- Kendal’s Tau

In [ ]:
cor3 = df.corr(method="kendall")

# [DAS509] 7강 자료의 재표현

# 자료의 재표현

- 변수변환 - 새로운 척도에서의 표현
- 데이터의 분포가 지나치게 한쪽으로 기울어져 있으면 분석에 문제가 발생할 가능성이 높음
- 이러한 문제를 해결해주기 위한 방법으로 활용
- 통계학에서는 주로 데이터의 분포를 대칭화 혹은 나아가 정규화하기 위한 목적으로 많이 활용

In [ ]:
mammals = pd.read_csv("mammals.csv", index_col = 0)
mammals

- 두변수 간의 관계를 살펴보자

In [ ]:
mammals.plot.scatter('body', 'brain', alpha = 0.3) # scatter plt

- 두변수 모두 한쪽으로 쏠려 있음을 알 수 있다.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(8, 4), layout='constrained')

axes[0].hist(mammals.body)
axes[0].set_title("Body")
axes[1].hist(mammals.brain)
axes[1].set_title("Brain")
fig.suptitle("Raw_data")

- 로그변환을 시도해 보자

In [ ]:
mammals['body_log'] = np.log(mammals['body'])
mammals['brain_log'] = np.log(mammals['brain'])
mammals

mammals.plot.scatter('body_log', 'brain_log', alpha = 0.5, title = "log_transformation")

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(6,4), layout='constrained')

axes[0].hist(mammals.body_log)
axes[0].set_title("Body")
axes[1].hist(mammals.brain_log)
axes[1].set_title("Brain")
fig.suptitle("Log_Transformed_data")

# 대표적인 변수변환의 종류

- 로그 (Log)
- 제곱근 (Square Root)
- 거듭곱 (Power) 변환
- Box-Cox 변환
    
    $$
    x^{(\lambda)} = \left\{\begin{array}{ll} \frac{x^\lambda - 1}{\lambda}, & \lambda \neq 0\\ \log x, & \lambda = 0\end{array}\right.
    $$
    
    - $\lambda$에 관계없이 $x=1$에서의 값이 0 ( + 미분계수가 1)
    - 로그변환과 거듭곱 변환의 일반화 형태
    - 데이터를 정규분포와 가장 유사하게 만드는 $\lambda$값을 찾을 수 있음
    - scipy 라이브러리 내의 stats모듈에 boxcox 함수 활용

In [ ]:
from scipy import stats

mammals['body_box'], lambda_body_box = stats.boxcox(mammals.body)
mammals['brain_box'], lambda_brain_box = stats.boxcox(mammals.brain)

In [ ]:
lambda_body_box, lambda_brain_box

In [ ]:
mammals.plot.scatter('body_box', 'brain_box', alpha = 0.5, 
                                              title = "boxcox_transformation")


# Feature Engineering

- 기계학습에서 최적의 학습 성능을 내기 위해 데이터를 적절히 변화하는 것을 의미 (새로운 용어일 뿐 기본적으로 변수 변환)
- 기본적으로 Scaling을 의미
    - 표준화 (Standardization): 각 변수별로 평균이 0, 분산이 1이 되도록 변환
    
    $$ z = \frac{x - m_x}{s_x} $$

In [ ]:
df = pd.DataFrame([
    [2, 1, 3],
    [3, 2, 5],
    [3, 4, 7],
    [5, 5, 10],
    [7, 5, 12],
    [2, 5, 7],
    [8, 9, 13],
    [9, 10, 13],
    [6, 12, 12],
    [9, 2, 13],
    [6, 10, 12],
    [2, 4, 6]
], columns=['hour', 'attendance', 'score'])

# stadardization: 방법1
df['hour_std'] = (df['hour'] - np.mean(df['hour'])) / np.std(df['hour'])

In [ ]:
# 방법2
import scipy.stats as ss 

df_std = ss.zscore(df)
df_std

In [ ]:
# 방법3
import sklearn
from sklearn.preprocessing import *

df_std = StandardScaler().fit_transform(df)
pd.DataFrame(df_std)

- Min-Max Scaling: 최대값과 최소값을 각각 1과 0으로 맞추는 변환

$$
u = \frac{x-\min(x)}{\max(x) - \min(x)}
$$

In [ ]:
# 방법1
mn = np.min(df['hour'])
mx = np.max(df['hour'])
df['hour_minmax'] = (df['hour'] - mn) / (mx - mn)

In [ ]:
# 방법2
df_mx = MinMaxScaler().fit_transform(df)
pd.DataFrame(df_mx)

- Robust 표준화: 일반적인 표준화와 달리, 중위수와 IQR을 이용

$$
z = \frac{x - \text{Median}_x}{IQR}
$$

In [ ]:
# 방법1
q1, q2, q3 = df.iloc[:,0].quantile([0.25, 0.5, 0.75])
df['hour_robust'] = (df['hour'] - q2) / (q3 - q1)

In [ ]:
# 방법2
df_rb = RobustScaler().fit_transform(df)
pd.DataFrame(df_rb)

- 다변량 정규화 / 백색잡음화: 모든 변수에 대해 한꺼번에 다변량 정규화를 실시하여 평균이 0, 분산이 1이 될 뿐만아니라, 변수간의 상관관계도 제거하는 변환
$$
\mathbf{z} = \mathbf{\Sigma_x}^{-1/2}(\mathbf{x} - \mathbf{m}_x)
$$

In [ ]:
data_raw = df.iloc[:,0:3]
data_cen = data_raw - data_raw.mean() # centering

# uncorrleated
cov_array = data_cen.cov()
precision = np.linalg.inv(cov_array)
cov_object = stats.Covariance.from_precision(precision)

z = cov_object.whiten(data_cen) # final result - array

In [ ]:
pd.DataFrame(z).mean() # all 0
pd.DataFrame(z).cov() # Identity matrix